# Autonomous News Reporter
Analyses the current news by region or topic, summerises it and sends a notification depending on importance and region affected.



In [23]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime

load_dotenv(override=True)
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

### World News MCP Server
MCP server to gather current world news from different sources.

In [24]:
# News MCP tools
display(Markdown(f"### News MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@newsmcp/server"]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

# Fetch News MCP tools
display(Markdown(f"### Browser MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")



### News MCP tools

get_news - Get top news events happening in the world right now. Returns AI-clustered, deduplicated news stories ranked by importance. Present results as a multi-story news briefing — cover the top events, not just one. Each event should be 1-2 lines with its summary and 1-2 source links. Prioritize suppressing rich link cards/previews over pretty link labels. For Discord-style clients, output source URLs directly as '<https://...>' and do not use masked markdown links. Never emit raw standalone URLs outside no-preview wrappers. Only deep-dive into a specific event if the user asks for detail.
get_news_detail - Get full details about a specific news event including context and all source articles. Use an event ID from get_news results. Include source article URLs so the user can read original reporting, but prioritize suppressing rich link cards/previews. For Discord-style clients, output source URLs directly as '<https://...>' and avoid masked markdown links. Never emit raw standalone

### Browser MCP tools

brave_web_search - Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. 
brave_local_search - Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, restaurants, services, etc. Returns detailed information including:
- Business names and addresses
- Ratings and review counts
- Phone numbers and opening hours
Use this when the query implies 'near me' or mentions specific locations. Automatically falls back to web search if no local results are found.


In [ ]:
NEWS_REPORTER_MODEL = "gpt-4o-mini"

world_news_mcp_params = [
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env},
    {"command": "npx", "args": ["-y", "@newsmcp/server"]}
]

world_news_mcp_servers = [MCPServerStdio(params) for params in world_news_mcp_params]

# News Reporter Agent
async def get_news_reporter(mcp_servers) -> Agent:
    instructions = f"""
    You are a news reporter. You are able to get the latest news about interesting topics or regions from the world.
    Search for at most 3 news articles and summarize them in 2-3 paragraphs. Concetrate mostly on the latest
    news and the most important details.
    Please take your time to make multiple searches to get a comprehensive overview, and then summarize your findings.
    Also, please verify the sources of the news to ensure the accuracy and credibility of the information.
    """
    news_reporter = Agent(
        name="News Reporter",
        instructions=instructions,
        model=NEWS_REPORTER_MODEL,
        mcp_servers=mcp_servers
    )
    return news_reporter

In [27]:
async def get_news_reporter_tool(mcp_servers) -> Tool:
    news_reporter = await get_news_reporter(mcp_servers)
    return news_reporter.as_tool(
        tool_name="get_news_reporter",
        tool_description="Get the latest news about interesting topics or regions from the world.",
        tool_func=get_news_reporter(mcp_servers)
    )

    


In [29]:
news_question = "What's the latest breaking news in Africa?"

for server in world_news_mcp_servers:
    await server.connect()
news_reporter = await get_news_reporter(world_news_mcp_servers)
with trace("News Reporter"):
    result = await Runner.run(news_reporter, news_question, max_turns=30)
display(Markdown(result.final_output))

Here's the latest breaking news from Africa:

1. **Economic Impact of Middle East War**: A new report by the African Union and the African Development Bank warns that the ongoing conflict in the Middle East poses significant risks to the African economy. If the war extends beyond six months, Africa could see a 0.2 percentage point loss in GDP for 2026, mainly due to soaring food and fuel prices, as well as increased shipping costs. The report highlights that currencies in 29 African countries have already depreciated, complicating debt servicing and import costs. This situation underscores the intertwining global economic challenges stemming from regional conflicts. For further readings, you can check sources like Asharq Al-Awsat and Al-Monitor.

2. **Senegal's Measures Against Rising Costs**: In response to the skyrocketing oil prices driven by the Iran war, Senegal's government has mandated a ban on non-essential foreign trips for ministers. Prime Minister Ousmane Sonko indicated that the country's budget was estimated on an oil price of $62 per barrel, which has now nearly doubled, pressuring the national economy. This directive aims to reduce public spending amid an escalating energy crisis affecting household and business costs. More information can be found in articles from BBC and the South China Morning Post.

3. **Challenges for South African Farmers**: Farmers in South Africa's Western Cape are facing serious hurdles for the upcoming winter crop season due to rising costs of fuel and fertilizers directly linked to the conflict in the Middle East. After a challenging previous season, the farmers are grappling with increased input costs, which may necessitate policy discussions regarding domestic wheat import tariffs to help support them. However, it is expected that consumers will not feel immediate impacts due to sufficient global wheat supplies. For additional context, resources such as Wandile Sihlobo’s analysis provide deeper insight.

These developments reflect the interconnected nature of global conflicts and their repercussions on local economies across Africa.